# NB9 — RF-DETR + FashionCLIP Core-7 Detection V1

Notebook smoke-test cho implementation trong `src/detection`. Mục tiêu không phải so sánh detector nữa; RF-DETR được dùng để lấy garment boxes, còn `coarse_category` được dự đoán trực tiếp bằng cosine similarity giữa **FashionCLIP image embedding** và bảy text prototypes Core-7.

Canonical flow:

```text
image -> RF-DETR -> crop -> FashionCLIP 512-d L2 -> Core-7 cosine -> scorer handoff
```

`master_category` không được suy diễn cho ảnh user.


## 1. Runtime

Notebook tự clone/checkout đúng feature branch và cài dependency detection. NB9 pin `numpy<2.4` vì NumPy 2.4.x hiện gây lỗi `_blas_supports_fpe` trong SciPy/NumPy import chain trên Colab. Nếu pip phải thay NumPy sau khi NumPy đã được load trong kernel, cell install sẽ yêu cầu restart runtime trước khi chạy inference.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import importlib

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "feat/detection-rfdetr-fashionclip-core7"
REPO_DIR = Path("/content/opisoverated")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

os.chdir(REPO_DIR)
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        sys.modules.pop(module_name, None)

current_branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
print(f"Current working directory: {os.getcwd()}")
print(f"Git branch: {current_branch}")


In [ ]:
# Install the branch's detection runtime dependencies.
# Keep pip output visible so resolver/runtime compatibility errors remain diagnosable.
from importlib import metadata as importlib_metadata

requirements_path = REPO_DIR / "requirements-detection.txt"
print(f"Python: {sys.version.split()[0]}")
print(requirements_path.read_text(encoding="utf-8"))

numpy_loaded_before = "numpy" in sys.modules
numpy_runtime_before = None
if numpy_loaded_before:
    numpy_runtime_before = getattr(sys.modules["numpy"], "__version__", None)

try:
    numpy_dist_before = importlib_metadata.version("numpy")
except importlib_metadata.PackageNotFoundError:
    numpy_dist_before = None

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "-r",
        str(requirements_path),
    ],
    check=True,
)

numpy_dist_after = importlib_metadata.version("numpy")
print(f"numpy(dist)=={numpy_dist_after}")

# A binary package cannot be safely replaced underneath an already-imported NumPy.
# Stop early with an explicit restart instruction instead of failing later inside SciPy.
if numpy_loaded_before and numpy_runtime_before != numpy_dist_after:
    raise RuntimeError(
        "NumPy changed on disk while an older NumPy is already loaded in this kernel "
        f"(runtime={numpy_runtime_before}, installed={numpy_dist_after}). "
        "Restart the Colab runtime, then Run all again."
    )

major_minor = tuple(int(part) for part in numpy_dist_after.split(".")[:2])
if major_minor >= (2, 4):
    raise RuntimeError(
        f"Unsupported NumPy for NB9: {numpy_dist_after}. "
        "Expected numpy<2.4 because NumPy 2.4.x currently breaks the "
        "SciPy/RF-DETR import chain on Colab."
    )

for package_name in ("torch", "numpy", "transformers", "rfdetr", "huggingface_hub"):
    try:
        print(f"{package_name}=={importlib_metadata.version(package_name)}")
    except importlib_metadata.PackageNotFoundError:
        print(f"{package_name}: NOT INSTALLED")

print("Detection dependencies installed.")


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "detection" / "config.py").exists():
    raise RuntimeError(
        f"Detection implementation is missing in {REPO_ROOT}. "
        f"Make sure branch {BRANCH!r} is checked out by rerunning the clone/checkout cell."
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "configs/detection_rfdetr_fashionclip_core7_v1.json"
OUTPUT_ROOT = REPO_ROOT / "outputs/detection_v1"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing detection config: {CONFIG_PATH}")
print("repo:", REPO_ROOT)
print("config:", CONFIG_PATH)


## 2. Chọn ảnh

Mặc định notebook dùng `tests/animage.jpg` làm smoke image đã commit trong branch. Có thể thay `IMAGE_PATH` bằng ảnh khác, nhưng missing image phải hard-fail để không tạo false PASS.


In [ ]:
IMAGE_PATH = REPO_ROOT / "tests/animage.jpg"
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f"Smoke image not found: {IMAGE_PATH}")
print("image:", IMAGE_PATH)


## 3. Load versioned config


In [ ]:
from src.detection import load_detection_config

config = load_detection_config(CONFIG_PATH)
print(config)


## 4. Run detection

RF-DETR labels chỉ filter garment object/part. Category cuối cùng do FashionCLIP zero-shot Core-7 quyết định.


In [ ]:
from src.detection import DetectionPipeline

pipeline = DetectionPipeline(config)
result, image = pipeline.run(IMAGE_PATH)

print("accepted garments:", len(result.garments))
print("rejected detections:", len(result.rejected_detections))
for garment in result.garments:
    print(
        garment.candidate.detector_label,
        "->",
        garment.category.coarse_category,
        f"sim={garment.category.similarity:.4f}",
        f"margin={garment.category.margin:.4f}",
    )


## 5. Save crops + scorer inputs


In [ ]:
from src.detection.pipeline import save_detection_result

run_dir = OUTPUT_ROOT / IMAGE_PATH.stem
saved = save_detection_result(
    result,
    image,
    run_dir,
    scorer_min_items=config.scorer_min_items,
    scorer_max_items=config.scorer_max_items,
)
print(saved)


## 6. Inspect metadata

`detection_result.json` giữ detector confidence, box, Core-7 cosine và margin nhưng **không tạo `master_category` giả**.


In [ ]:
import json

metadata_path = run_dir / "detection_result.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
print(json.dumps(metadata, ensure_ascii=False, indent=2)[:8000])


## 7. Visualize crops


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

crop_paths = sorted((run_dir / "crops").glob("*.jpg"))
for crop_path in crop_paths:
    plt.figure(figsize=(3, 3))
    plt.imshow(Image.open(crop_path))
    plt.title(crop_path.stem)
    plt.axis("off")
    plt.show()


## Acceptance gate

NB9 chỉ PASS khi inference thực sự chạy, có ít nhất một garment hợp lệ, embedding đúng 512-d, category thuộc Core-7, metadata được ghi và scorer handoff được tạo thành công cho smoke image.


In [ ]:
from pathlib import Path
from src.detection import CORE7_CATEGORY_TO_ID

assert len(result.garments) > 0, "NB9 FAIL: detector/classifier produced no accepted garment"
for garment in result.garments:
    assert len(garment.embedding) == 512, "NB9 FAIL: embedding is not 512-d"
    assert garment.category.coarse_category in CORE7_CATEGORY_TO_ID, "NB9 FAIL: invalid Core-7 category"

assert metadata_path.is_file(), "NB9 FAIL: detection_result.json missing"
assert saved["scorer_handoff_error"] is None, f"NB9 FAIL: {saved['scorer_handoff_error']}"
assert saved["scorer_inputs_path"] is not None, "NB9 FAIL: scorer_inputs.pt was not created"
assert Path(saved["scorer_inputs_path"]).is_file(), "NB9 FAIL: scorer_inputs.pt path missing"

print("NB9 SMOKE PASS")
